# Closed-Loop Data Center Cooling Control over BACnet/IP

*Newman, H. Michael. Bacnet: the global standard for building automation and control networks. Momentum Press, 2013.*

*Backer, James, C. Hübner, Thomas Hansemann, and Hermann Merz. Building Automation: Communication Systems with EIB/KNX, LON and BACnet. Springer, 2018.*

*Wang, Shengwei. Intelligent buildings and building automation. Routledge, 2009.*

Here we implement a closed-loop building automation system (BAS) simulation of a data center thermal management loop communicating over BACnet/IP (ASHRAE Standard 135). Rather than relying on raw register polling as in industrial Modbus TCP setups, the system models a field-level CRAC (Computer Room Air Conditioning) controller and a supervisory BMS (Building Management System) controller as independent BACnet/IP network nodes exchanging typed, self-describing engineering objects over UDP:

1. **Virtual CRAC Field Controller (`bacnet_twin.py` — `Device 1001`, `127.0.0.1:47808`):** Simulates the server hall's air-side energy balance, samples stochastic IT heat loads from `ITLoadSimulator` (`thermal_load.py`), evaluates a local high-temperature safety interlock (`OverheatAlarm` at $> 35.0^\circ\text{C}$), and serves live BACnet objects.
2. **Supervisory BMS PID Controller (`bacnet_pid.py` — `Device 2001`, `127.0.0.1:47809`):** Polls outlet air temperature and IT power at $1\text{ Hz}$ via BACnet `ReadProperty`, computes a discrete PID control law with integral anti-windup, and commands CRAC fan speed via `WriteProperty` at **BACnet Priority 8**.
3. **Supervisory BMS MPC Controller (`bacnet_mpc.py` — `Device 2002`, `127.0.0.1:47809`):** Replaces the fixed-gain PID controller with a receding-horizon Model Predictive Controller that uses the nonlinear air-side energy balance, feedforward IT load measurements, hard actuator bounds, and a soft safety barrier below the $35.0^\circ\text{C}$ alarm limit to optimize fan speed over a $10\text{ s}$ prediction horizon.

Transitioning from Modbus TCP to BACnet/IP introduces three fundamental building-automation abstractions:

* **Object-Oriented Data Model & Native Engineering Units:** Modbus exposes flat arrays of unnamed 16-bit integers (requiring manual $\times 100$ fixed-point scaling for temperatures). BACnet models equipment as standardized objects (`AnalogInput`, `AnalogValue`, `BinaryValue`) whose properties natively encode IEEE-754 `Real` floating-point values, human-readable names, descriptions, status flags, and ASHRAE engineering units (`degreesCelsius`, `kilowatts`, `percent`).
* **Standardized Application Services (`ReadProperty` / `WriteProperty`):** Instead of reading/writing numeric register addresses, supervisory controllers query and command objects by `(ObjectType, InstanceNumber)` and `PropertyIdentifier` (e.g., reading `presentValue` of `analogInput,1`).
* **16-Level Command Priority Array & Fail-Safe Relinquishment:** Commandable BACnet objects (`CommandableAnalogValueObject`) maintain a 16-slot priority array (`1` = Manual Life Safety, `8` = Manual Operator / Supervisory Override, `16` = Lowest) alongside a `relinquishDefault` property. The highest-priority non-null slot determines `presentValue`. When `bacnet_pid.py` or `bacnet_mpc.py` exits, its `finally:` block sends a `WriteProperty` request with `Null(())` at Priority 8, relinquishing control so the CRAC fan safely reverts to `relinquishDefault = 50.0%` rather than freezing at its last commanded speed.

| Object Identifier | Object Name | Object Class | Units / States | Role & Behavior |
| :--- | :--- | :--- | :--- | :--- |
| `analogInput,1` | `OutletAirTemp` | `AnalogInputObject` | `degreesCelsius` | Hot-aisle return air temperature sensor ($T_k$) |
| `analogInput,2` | `ITServerLoad` | `AnalogInputObject` | `kilowatts` | Total IT server hall heat load ($\dot{Q}_{\text{IT},k}$) |
| `analogValue,1` | `CRACFanSpeedCmd` | `CommandableAnalogValueObject` | `percent` | Commanded VFD fan speed ($u_k \in [20, 100]\%$; `relinquishDefault = 50.0`) |
| `binaryValue,1` | `OverheatAlarm` | `BinaryValueObject` | `active` / `inactive` | Field-level safety interlock (`active` when $T_k > 35.0^\circ\text{C}$) |

At each $1\text{ s}$ simulation tick, `bacnet_twin.py` samples the IT server heat load $\dot{Q}_{\text{IT},k}$ (combining a $500\text{ kW}$ baseline, a $100\text{ kW}$ sinusoidal diurnal cycle, $\pm 50\text{ kW}$ uniform noise, and stochastic $+200\text{ kW}$ batch-job spikes) and computes the air mass flow rate $\dot{m}_k$ from the active fan command $u_k \in [0, 100]\%$:

$$\dot{m}_k = \left(V_{\max}\frac{u_k}{100} + 0.5\right)\rho_{\text{air}}$$

where $V_{\max} = 50.0\text{ m}^3/\text{s}$, $\rho_{\text{air}} = 1.225\text{ kg/m}^3$, and the $0.5\text{ m}^3/\text{s}$ offset accounts for bypass/infiltration airflow. The instantaneous equilibrium outlet temperature is $T_{\text{eq},k} = T_{\text{amb}} + \frac{\dot{Q}_{\text{IT},k}}{\dot{m}_k c_p}$ (with $T_{\text{amb}} = 20.0^\circ\text{C}$ and $c_p = 1.006\text{ kJ/(kg}\cdot\text{K)}$), filtered by a first-order thermal inertia factor $\alpha = 0.2$:

$$T_{k+1} = T_k + \alpha \left( T_{\text{amb}} + \frac{\dot{Q}_{\text{IT},k}}{\left(V_{\max}\frac{u_k}{100} + 0.5\right)\rho_{\text{air}} c_p} - T_k \right)$$

* **Physical Cooling Bounds:** At $\dot{Q}_{\text{IT}} = 500\text{ kW}$, maximum cooling ($u = 100\%$) yields a steady-state outlet temperature of $28.03^\circ\text{C}$, while the uncommanded fail-safe speed (`relinquishDefault = 50%`) settles at $35.91^\circ\text{C}$ (tripping `OverheatAlarm`). Holding the target setpoint $T_{\text{set}} = 29.5^\circ\text{C}$ at $500\text{ kW}$ requires a steady-state fan speed of $u_{\text{ss}} = 84.42\%$.
* **Severe Actuator Nonlinearity:** Differentiating the discrete-time state equation with respect to fan command $u_k$ yields:
  $$\frac{\partial T_{k+1}}{\partial u_k} = -\frac{\alpha\,\dot{Q}_{\text{IT},k}\,(V_{\max}/100)}{\left(V_{\max}\frac{u_k}{100} + 0.5\right)^2 \rho_{\text{air}} c_p}$$
  At $\dot{Q}_{\text{IT}} = 500\text{ kW}$, the local process gain shifts from **$-0.3680^\circ\text{C}/\%$** at $u = 20\%$ to **$-0.0159^\circ\text{C}/\%$** at $u = 100\%$—a **$23.13\times$ reduction in actuator sensitivity** at high airflow.

`bacnet_pid.py` implements a single-input, single-output (SISO) feedback law acting solely on the temperature error $e_k = T_k - T_{\text{set}}$ ($T_{\text{set}} = 29.5^\circ\text{C}$, $K_p = 12.0$, $K_i = 0.5$, $K_d = 0.1$):

$$u_k = \text{clip}\!\left(K_p e_k + K_i I_k + K_d \frac{e_k - e_{k-1}}{\Delta t},\; 20.0,\; 100.0\right), \quad I_k = \text{clip}\!\left(I_{k-1} + e_k \Delta t,\; -50.0,\; 50.0\right)$$

* **Strengths:** Simple, model-free feedback with clamped integral anti-windup and actuator saturation.
* **Limitations:** Purely reactive (it only responds *after* a $+200\text{ kW}$ batch spike raises $T_k$) and uses fixed gains across a $23.1\times$ nonlinear gain range.

`bacnet_mpc.py` embeds the exact nonlinear thermal model inside a receding-horizon optimal control problem over a prediction horizon of $N = 10\text{ steps}$ ($10\text{ s}$). At every $1\text{ Hz}$ control cycle, it solves:

$$\min_{u_0, \dots, u_{N-1}} \sum_{k=0}^{N-1} \left[ W_{\text{track}}\left(\hat{T}_{k+1} - T_{\text{set}}\right)^2 + W_{\text{effort}}\left(u_k - u_{\text{ss}}\right)^2 + W_{\text{slew}}\left(u_k - u_{k-1}\right)^2 + W_{\text{safety}}\max\!\left(0,\, \hat{T}_{k+1} - T_{\text{safe}}\right)^2 \right]$$

subject to the discrete-time nonlinear dynamics and hard actuator constraints $20.0\% \le u_k \le 100.0\%$, with weights $W_{\text{track}} = 25.0$, $W_{\text{effort}} = 0.002$, $W_{\text{slew}} = 0.05$, $W_{\text{safety}} = 500.0$, and soft barrier threshold $T_{\text{safe}} = 34.5^\circ\text{C}$ ($0.5^\circ\text{C}$ below `OverheatAlarm`):

1. **Feedforward Disturbance Coupling & Online Observer:** `bacnet_mpc.py` reads `ITServerLoad` (`analogInput,2`) directly as a known thermal disturbance $\hat{Q}_{\text{IT}}$. If the power sensor is unavailable (`0.0`), it automatically falls back to an inverse-dynamics disturbance observer that infers $\hat{Q}_{\text{IT}}$ from the observed temperature transition $(T_k - T_{k-1})$ at $u_{k-1}$.
2. **Real-Time Nonlinear Optimization:** Warm-started at the analytical steady-state fan speed $u_{\text{ss}}(\hat{Q}_{\text{IT}}, T_{\text{set}})$, Sequential Least Squares Programming (`SLSQP`) solves the 10-variable nonlinear program in **$\sim 10\text{ ms}$**—two orders of magnitude faster than the $1\text{ s}$ sampling interval—and applies the optimal first move $u_0^*$ to `analogValue,1` at Priority 8.

| Feature | `bacnet_pid.py` (Supervisory PID) | `bacnet_mpc.py` (Supervisory NMPC) |
| :--- | :--- | :--- |
| **Control Paradigm** | Reactive SISO feedback on $T_k - T_{\text{set}}$ | Predictive MIMO optimal control using $T_k$ and $\dot{Q}_{\text{IT},k}$ |
| **Plant Nonlinearity** | Fixed linear gains ($K_p, K_i, K_d$) | Exact nonlinear air-side energy balance across $20\text{--}100\%$ flow |
| **Disturbance Rejection** | Waits for temperature error to accumulate | Immediate feedforward fan ramp upon $+200\text{ kW}$ load spikes |
| **Constraint Handling** | Post-hoc output clipping & integral clamping | Explicit box constraints ($[20, 100]\%$), slew-rate penalty, and $34.5^\circ\text{C}$ safety barrier |
| **BACnet Integration** | `ReadProperty` + `WriteProperty` (Priority 8) + `Null()` relinquish | `ReadProperty` + `WriteProperty` (Priority 8) + `Null()` relinquish |